# CcMart — Spark Structured Streaming Demo
**ITCS 6190/8190 Cloud Computing for Data Analysis**

Simulates a real-time clickstream using the raw `click_stream.csv` rows, consumes it with Spark Structured Streaming, performs a **stream-static join** with the product catalog, and runs two **windowed aggregations** to detect trending categories and high-intent sessions.

## Setup

In [1]:
import os, shutil

JAVA11 = "/usr/local/sdkman/candidates/java/11.0.30-ms"
os.environ["JAVA_HOME"] = JAVA11
clean_path = [p for p in os.environ["PATH"].split(":") if "sdkman" not in p and "jvm" not in p]
os.environ["PATH"] = f"{JAVA11}/bin:" + ":".join(clean_path)

# Driver memory must be set BEFORE the JVM starts (PYSPARK_SUBMIT_ARGS, not
# the builder .config()).
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 3g "
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, window, count, approx_count_distinct, from_json, to_timestamp,
    avg, round as spark_round, when, max as spark_max,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)

STREAM_INPUT = '../data/stream_input'
STREAM_CHECKPOINT = '../data/stream_checkpoint'

spark = SparkSession.builder \
    .appName('CcMart-Streaming') \
    .config('spark.sql.streaming.checkpointLocation', STREAM_CHECKPOINT) \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ.get("AWS_ACCESS_KEY_ID", "")) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ.get("AWS_SECRET_ACCESS_KEY", "")) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

BUCKET = os.environ.get("S3_BUCKET_PATH", "")
PROC = f"{BUCKET}/processed" if BUCKET else "../data/processed"

# Load clickstream — we sample from here for the simulator because the raw CSV
# lives on S3 and isn't pulled locally on this codespace.
clicks_src = spark.read.parquet(f'{PROC}/clickstream_clean')
print('Streaming session ready. Spark', spark.version)
print('Source rows available:', clicks_src.count())

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED
26/04/20 05:18:33 WARN Utils: Your hostname, codespaces-94354a resolves to a loopback address: 127.0.0.1; using 10.0.12.148 instead (on interface eth0)
26/04/20 05:18:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/codespace/.ivy2/cache
The jars for the packages stored in: /home/codespace/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6288cc7b-f9fd-4798-8c05-2f46940ccd1f;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 464ms :: artifacts dl 22ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnld

Streaming session ready. Spark 3.5.1


Source rows available: 12833602


## Define Clickstream Event Schema
Matches the raw `click_stream.csv` schema (before ingestion). `event_metadata` is a JSON-ish string that we parse into structured sub-fields.

In [2]:
EVENT_SCHEMA = StructType([
    StructField('session_id',      StringType(), True),
    StructField('event_name',      StringType(), True),
    StructField('event_time',      StringType(), True),
    StructField('event_id',        StringType(), True),
    StructField('traffic_source',  StringType(), True),
    StructField('event_metadata',  StringType(), True),
])

# event_metadata is a JSON string with heterogeneous keys across event types.
META_SCHEMA = StructType([
    StructField('product_id',      IntegerType(), True),
    StructField('quantity',        IntegerType(), True),
    StructField('item_price',      DoubleType(),  True),
    StructField('search_keywords', StringType(),  True),
    StructField('promo_code',      StringType(),  True),
    StructField('payment_status',  StringType(),  True),
])

print('Event schema  :', [f.name for f in EVENT_SCHEMA.fields])
print('Meta  schema  :', [f.name for f in META_SCHEMA.fields])

Event schema  : ['session_id', 'event_name', 'event_time', 'event_id', 'traffic_source', 'event_metadata']
Meta  schema  : ['product_id', 'quantity', 'item_price', 'search_keywords', 'promo_code', 'payment_status']


## Simulate a Real-Time Event Stream
Sample rows from `clickstream_clean` (the raw CSV isn't pulled locally on the codespace), reconstruct the single-quoted Python-dict `event_metadata` from the flat `cs_*` columns so the consumer sees the same shape as the real raw feed, then drip-feed JSON batches. Spark's file-source streaming reader picks each batch up as a micro-batch.

In [ ]:
import json, threading, time
from datetime import datetime

SAMPLE_SIZE = 1500   # total events to simulate
BATCH_SIZE  = 25     # events per JSON file
DELAY_MS    = 200    # pause between batches

# Pull a sample to the driver. We fetch more rows than we need (some will be
# event types with sparse metadata — still valid events).
sample_rows = (clicks_src
    .select('session_id', 'event_name', 'event_time', 'event_id',
            'traffic_source',
            'cs_product_id', 'cs_quantity', 'cs_item_price',
            'search_keywords',
            'cs_promo_code', 'cs_promo_amount', 'cs_payment_status')
    .limit(SAMPLE_SIZE)
    .collect())
print(f'Collected {len(sample_rows)} sample rows for the simulator.')

def _reconstruct_meta(r):
    """Rebuild the single-quoted Python-dict event_metadata from cs_* cols."""
    d = {}
    if r['cs_product_id']     is not None: d['product_id']      = r['cs_product_id']
    if r['cs_quantity']       is not None: d['quantity']        = r['cs_quantity']
    if r['cs_item_price']     is not None: d['item_price']      = r['cs_item_price']
    if r['search_keywords']   is not None: d['search_keywords'] = r['search_keywords']
    if r['cs_promo_code']     is not None: d['promo_code']      = r['cs_promo_code']
    if r['cs_promo_amount']   is not None: d['promo_amount']    = r['cs_promo_amount']
    if r['cs_payment_status'] is not None: d['payment_status']  = r['cs_payment_status']
    return str(d)  # single-quoted — consumer normalises with regexp_replace

def stream_simulator():
    if os.path.exists(STREAM_INPUT):
        shutil.rmtree(STREAM_INPUT)
    os.makedirs(STREAM_INPUT, exist_ok=True)

    written, batch_num = 0, 0
    batch = []
    for r in sample_rows:
        evt_time = r['event_time']
        batch.append({
            'session_id':     r['session_id'] or '',
            'event_name':     r['event_name'] or '',
            'event_time':     evt_time.isoformat() if isinstance(evt_time, datetime) else str(evt_time or ''),
            'event_id':       r['event_id'] or '',
            'traffic_source': r['traffic_source'] or '',
            'event_metadata': _reconstruct_meta(r),
        })
        if len(batch) >= BATCH_SIZE:
            batch_num += 1
            with open(os.path.join(STREAM_INPUT, f'events_{batch_num:06d}.json'), 'w') as o:
                for evt in batch:
                    o.write(json.dumps(evt) + '\n')
            written += len(batch)
            batch = []
            time.sleep(DELAY_MS / 1000.0)
    if batch:
        batch_num += 1
        with open(os.path.join(STREAM_INPUT, f'events_{batch_num:06d}.json'), 'w') as o:
            for evt in batch:
                o.write(json.dumps(evt) + '\n')
        written += len(batch)
    print(f'Simulator done: {written} events in {batch_num} files → {STREAM_INPUT}')

# Kick off on a background thread so the stream consumer below can start reading
# files while the producer is still writing them.
threading.Thread(target=stream_simulator, daemon=True).start()
print('Simulator thread started.')

Collected 1500 sample rows for the simulator.
Simulator thread started.


Simulator done: 1500 events in 60 files → ../data/stream_input


## Stream-Static Join + Parsed Event Payload
Read streaming JSON batches, parse the single-quoted `event_metadata` string into structured columns, and **join** against the static `products_clean` Parquet (small → auto-broadcast).

In [4]:
BUCKET = os.environ.get("S3_BUCKET_PATH", "")
PROC   = f"{BUCKET}/processed" if BUCKET else "../data/processed"

# Static side of the join — product catalog for enrichment.
products_static = spark.read.parquet(f'{PROC}/products_clean') \
    .select('product_id', 'masterCategory', 'subCategory', 'articleType')

# Streaming source.
raw_stream = spark.readStream \
    .schema(EVENT_SCHEMA) \
    .option('maxFilesPerTrigger', 5) \
    .json(STREAM_INPUT)

# event_metadata uses single-quoted Python-dict syntax in the raw CSV; normalize
# it to valid JSON before from_json().
parsed = raw_stream \
    .withColumn('meta_json',  F.regexp_replace('event_metadata', "'", '"')) \
    .withColumn('meta',       from_json('meta_json', META_SCHEMA)) \
    .withColumn('event_ts',   to_timestamp('event_time')) \
    .withColumn('product_id', col('meta.product_id')) \
    .withColumn('quantity',   col('meta.quantity')) \
    .withColumn('item_price', col('meta.item_price'))

enriched = parsed.join(products_static, on='product_id', how='left')

print('isStreaming:', enriched.isStreaming)
enriched.printSchema()

isStreaming: True
root
 |-- product_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- event_metadata: string (nullable = true)
 |-- meta_json: string (nullable = true)
 |-- meta: struct (nullable = true)
 |    |-- product_id: integer (nullable = true)
 |    |-- quantity: integer (nullable = true)
 |    |-- item_price: double (nullable = true)
 |    |-- search_keywords: string (nullable = true)
 |    |-- promo_code: string (nullable = true)
 |    |-- payment_status: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- item_price: double (nullable = true)
 |-- masterCategory: string (nullable = true)
 |-- subCategory: string (nullable = true)
 |-- articleType: string (nullable = true)



## Windowed Aggregations (5-minute tumbling windows, 10-min watermark)

**Query 1 — category trending:** events per `masterCategory` per 5-minute window with active-session counts.

**Query 2 — session engagement:** per-session event count, unique actions, and cart/booking flags — the inputs a live "is this user about to bounce?" model would consume.

In [5]:
category_trending = (enriched
    .filter(col('masterCategory').isNotNull())
    .withWatermark('event_ts', '10 minutes')
    .groupBy(window('event_ts', '5 minutes'), col('masterCategory'))
    .agg(
        count('*').alias('event_count'),
        approx_count_distinct('session_id').alias('active_sessions'),
    )
)

session_engagement = (enriched
    .withWatermark('event_ts', '10 minutes')
    .groupBy(window('event_ts', '5 minutes'),
             col('session_id'), col('traffic_source'))
    .agg(
        count('*').alias('event_count'),
        approx_count_distinct('event_name').alias('unique_actions'),
        spark_max(when(col('event_name') == 'ADD_TO_CART', 1).otherwise(0)).alias('has_cart'),
        spark_max(when(col('event_name') == 'BOOKING', 1).otherwise(0)).alias('has_booking'),
    )
)

print('Queries defined: category_trending, session_engagement')

Queries defined: category_trending, session_engagement


## Start the Stream + Collect Results
Q1 → console sink so you can see micro-batches as they arrive.
Q2 → memory sink so the notebook can query it with regular Spark SQL once the run ends.

Runs for ~45 s, then stops gracefully.

In [6]:
if os.path.exists(STREAM_CHECKPOINT):
    shutil.rmtree(STREAM_CHECKPOINT)

q1 = (category_trending.writeStream
      .outputMode('update')
      .format('console')
      .option('truncate', False)
      .option('numRows', 20)
      .queryName('category_trending')
      .start())

q2 = (session_engagement.writeStream
      .outputMode('update')
      .format('memory')
      .queryName('session_scores')
      .start())

print('Active queries:', [q.name for q in spark.streams.active])

try:
    q1.awaitTermination(45)
except Exception as e:
    print('Stream terminated:', e)
finally:
    for q in spark.streams.active:
        q.stop()
    print('All streams stopped.')

26/04/20 05:20:48 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/20 05:20:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Active queries: ['session_scores', 'category_trending']


26/04/20 05:20:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+--------------+-----------+---------------+
|window                                    |masterCategory|event_count|active_sessions|
+------------------------------------------+--------------+-----------+---------------+
|{2020-02-05 15:30:00, 2020-02-05 15:35:00}|Footwear      |1          |1              |
|{2020-02-03 04:00:00, 2020-02-03 04:05:00}|Accessories   |1          |1              |
|{2022-06-09 04:45:00, 2022-06-09 04:50:00}|Apparel       |1          |1              |
|{2020-02-02 16:10:00, 2020-02-02 16:15:00}|Apparel       |1          |1              |
|{2022-02-03 18:15:00, 2022-02-03 18:20:00}|Apparel       |1          |1              |
|{2020-02-08 18:50:00, 2020-02-08 18:55:00}|Footwear      |1          |1              |
|{2020-02-08 22:45:00, 2020-02-08 22:50:00}|Footwear      |1          |1              |
|{2019-09-22 07:40:00, 

All streams stopped.


26/04/20 05:21:35 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 1, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/04/20 05:21:35 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 1, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.


26/04/20 05:21:35 WARN TaskSetManager: Lost task 4.0 in stage 17.0 (TID 55) (8ab5ab07-6cbc-4c2a-ac88-284d0b35e04c.internal.cloudapp.net executor driver): TaskKilled (Stage cancelled: Job 12 cancelled part of cancelled job group 572b08a2-f220-4b17-90c6-f4fb0199ea10)
26/04/20 05:21:35 WARN TaskSetManager: Lost task 5.0 in stage 17.0 (TID 56) (8ab5ab07-6cbc-4c2a-ac88-284d0b35e04c.internal.cloudapp.net executor driver): TaskKilled (Stage cancelled: Job 12 cancelled part of cancelled job group 572b08a2-f220-4b17-90c6-f4fb0199ea10)


## Post-Stream Analysis — High-Intent & Anomalous Sessions
The memory sink `session_scores` is now a regular Spark table. Flag sessions with cart-adds, and detect anomalously active sessions (mean + 2σ event count) — exactly the signal the pitch's "real-time decisioning" engine would act on.

In [ ]:
scores = spark.sql("SELECT * FROM session_scores")
total_rows = scores.count()
print(f'Session-score rows captured: {total_rows}')

if total_rows > 0:
    print('\nHigh-intent sessions (has_cart = 1, no booking yet):')
    scores.filter((col('has_cart') == 1) & (col('has_booking') == 0)) \
          .select('session_id', 'traffic_source', 'event_count', 'unique_actions') \
          .show(15, truncate=False)

    stats = scores.agg(
        avg('event_count').alias('mean_events'),
        F.stddev('event_count').alias('std_events'),
    ).first()
    mean_events = stats['mean_events'] or 0
    std_events  = stats['std_events']  or 1
    threshold = mean_events + 2 * std_events
    print(f'\nAnomaly threshold (mean + 2σ): {threshold:.1f} events/session')

    scores.filter(col('event_count') > threshold) \
          .select('session_id', 'event_count', 'unique_actions', 'traffic_source') \
          .show(15, truncate=False)

Session-score rows captured: 246

High-intent sessions (has_cart = 1, no booking yet):
+------------------------------------+--------------+-----------+--------------+
|session_id                          |traffic_source|event_count|unique_actions|
+------------------------------------+--------------+-----------+--------------+
|c408f2e0-4998-4876-a4fe-13b6d139d5b2|MOBILE        |1          |1             |
|73441215-e0d6-47a2-b82c-07224e94d928|WEB           |1          |1             |
|f6a95f84-d3ac-4bfb-bb5b-bf8efb93d026|MOBILE        |1          |1             |
|f6a95f84-d3ac-4bfb-bb5b-bf8efb93d026|MOBILE        |1          |1             |
|11654dfc-aebd-4ae9-9ae0-beb41e7c68e1|MOBILE        |1          |1             |
|11654dfc-aebd-4ae9-9ae0-beb41e7c68e1|MOBILE        |1          |1             |
|01794c1d-27c8-4d49-b010-fec52a57031f|MOBILE        |2          |2             |
|595c049c-1f16-46d5-8222-761fd65021df|MOBILE        |1          |1             |
|f6a95f84-d3ac-4bfb-bb

: 

: 

## Concept: Why Structured Streaming?

| Batch (nightly) | Streaming (real-time) |
|---|---|
| React **next day** | React **while customer is on site** |
| Miss cart-abandon opportunity | Trigger 10% promo in same session |
| Report-driven | Action-driven |

Spark Structured Streaming treats data as an **unbounded table** — the same SQL you write for batch works on live data.

## Run the full streaming pipeline
```bash
python ../src/streaming.py
```
Outputs saved to `data/stream_output/`.

> Note: `src/streaming.py` still references an older sample-data schema (`add_to_cart` lowercase, `checkout` event) that doesn't match the real clickstream — the notebook above is the canonical working version.